# fixin bugs

# test pipeline

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
from flax import nnx


from gensbi.models import SimformerParams, FluxParams
from gensbi.recipes import SimformerFlowPipeline, SimformerDiffusionPipeline, FluxFlowPipeline, FluxDiffusionPipeline

import itertools

from gensbi.utils.model_wrapping import _expand_dims


nsamples = 1000
rng = jax.random.PRNGKey(0)

dim_theta = 2
dim_data = 7
dim_joint = dim_theta + dim_data


theta = jax.random.normal(rng, (nsamples, dim_theta, 1))
x = jax.random.normal(rng, (nsamples, dim_data, 1))

data = jnp.concatenate([theta, x], axis=1)

train_data = data[:800].reshape(10, -1, dim_joint, 1)
val_data = data[800:].reshape(10, -1, dim_joint, 1)

train_dataset = itertools.cycle(train_data)
val_dataset = itertools.cycle(val_data)


In [2]:
next(train_dataset).shape

(80, 9, 1)

In [3]:

params_simf = SimformerParams(
    rngs = nnx.Rngs(0),
    dim_value = 2,
    dim_id = 2, 
    dim_condition = 2, 
    dim_joint= dim_joint,
    fourier_features = 32,
    num_heads = 2,
    num_layers = 1,
    widening_factor = 3,
    qkv_features = 10, 
    num_hidden_layers = 1)

pipeline_smf_flow = SimformerFlowPipeline(
    train_dataset, val_dataset, dim_theta, dim_data, params_simf
)

pipeline_smf_diff = SimformerDiffusionPipeline(
    train_dataset, val_dataset, dim_theta, dim_data, params_simf
)

params_flux = FluxParams(
            in_channels=1,
            vec_in_dim=None,
            context_in_dim=1,
            mlp_ratio=1,
            qkv_multiplier=1,
            num_heads=2,
            depth=2,
            depth_single_blocks=2,
            axes_dim=[2,],
            use_rope = False,
            qkv_bias=True,
            obs_dim = dim_theta,
            cond_dim = dim_data,
            theta=20,
            rngs=nnx.Rngs(default=42),
            param_dtype=jnp.float32,
        )

pipeline_flux_flow = FluxFlowPipeline(
    train_dataset, val_dataset, dim_theta, dim_data, params_flux
)

pipeline_flux_diff = FluxDiffusionPipeline(
    train_dataset, val_dataset, dim_theta, dim_data, params_flux
)

def test_simformer_flow_training_step():
    pipeline_smf_flow.train(nnx.Rngs(0), nsteps=2)
    sample = pipeline_smf_flow.sample(jax.random.PRNGKey(1), jnp.arange(dim_data)[None,...], nsamples=32)
    assert sample.shape == (32, dim_theta), f"Expected shape (32, {dim_theta}), got {sample.shape}"

def test_simformer_diff_training_step():
    pipeline_smf_diff.train(nnx.Rngs(0), nsteps=2)
    sample = pipeline_smf_diff.sample(jax.random.PRNGKey(1), jnp.arange(dim_data)[None,...], nsamples=32)
    assert sample.shape == (32, dim_theta), f"Expected shape (32, {dim_theta}), got {sample.shape}"

def test_flux_flow_training_step():
    pipeline_flux_flow.train(nnx.Rngs(0), nsteps=2)
    sample = pipeline_flux_flow.sample(jax.random.PRNGKey(1), jnp.arange(dim_data)[None,...], nsamples=32)
    assert sample.shape == (32, dim_theta), f"Expected shape (32, {dim_theta}), got {sample.shape}"

def test_flux_diff_training_step():
    pipeline_flux_diff.train(nnx.Rngs(0), nsteps=2)
    sample = pipeline_flux_diff.sample(jax.random.PRNGKey(1), jnp.arange(dim_data)[None,...], nsamples=32)
    assert sample.shape == (32, dim_theta), f"Expected shape (32, {dim_theta}), got {sample.shape}"



In [4]:
test_simformer_flow_training_step()
test_flux_flow_training_step()
test_simformer_diff_training_step()

100%|██████████| 2/2 [00:03<00:00,  1.67s/it]


Training complete and model saved.


100%|██████████| 2/2 [00:07<00:00,  3.93s/it]


Training complete and model saved.


100%|██████████| 2/2 [00:03<00:00,  1.74s/it]


Training complete and model saved.


In [5]:

test_flux_diff_training_step()

100%|██████████| 2/2 [00:07<00:00,  3.81s/it]


Training complete and model saved.


AssertionError: Expected shape (32, 2), got (32, 2, 1)

In [6]:
pipe = pipeline_flux_diff

In [7]:
batch = pipe._next_batch()

In [8]:
batch.shape

(80, 9, 1)

In [8]:
from einops import repeat

pipe = pipeline_flux_diff
# pipe = pipeline_smf_diff
self = pipe
key = jax.random.PRNGKey(0)
batch = pipe._next_batch()


# obs = jnp.take_along_axis(batch, self.obs_ids, axis=1)
# cond = jnp.take_along_axis(batch, self.cond_ids, axis=1)
obs = batch[:, : self.dim_theta, ...]
cond = batch[:, self.dim_theta :, ...]

rng_x0, rng_sigma = jax.random.split(key, 2)

x_1 = obs
sigma = self.path.sample_sigma(rng_sigma, x_1.shape[0])
sigma = repeat(sigma, f"b -> b {'1 ' * (x_1.ndim - 1)}")
print(sigma.shape, x_1.shape)

batch = (x_1, sigma)
loss = self.loss_fn(rng_x0, pipe.model, batch, cond, self.obs_ids, self.cond_ids)
loss

(80, 1, 1) (80, 2, 1)


Array(6.541977, dtype=float32)

In [24]:
sde=self.path.scheduler

In [30]:
model_extras = {
            "cond": cond,
            "obs_ids": self.obs_ids,
            "cond_ids": self.cond_ids,
        }

In [31]:
condition_mask = 0
condition_value = 0
n_steps=12
S_min = 0.05
S_max = 50.0
S_churn = 0.0
S_noise = 1.0

# Time step discretization.
step_indices = jnp.arange(n_steps)

t_steps = sde.timesteps(step_indices, n_steps)
t_steps = jnp.append(t_steps, 0)

# Main sampling loop.
x_next = x_1 * t_steps[0]
i = 1


key, subkey = jax.random.split(key)
t_cur = t_steps[i]
t_next = t_steps[i+1]
x_curr = x_next

# Increase noise temporarily.
in_range = jnp.logical_and(t_cur >= S_min, t_cur <= S_max)
# print(in_range)
gamma = jax.lax.cond(in_range, lambda: jnp.minimum(S_churn / n_steps, jnp.sqrt(2) - 1), lambda: 0.0)
t_hat = t_cur + gamma * t_cur # sigma at the specific time step
sqrt_arg = jnp.clip(t_hat ** 2 - t_cur ** 2, min=0, max=None)
x_hat = x_curr + jnp.sqrt(sqrt_arg) * S_noise * jax.random.normal(subkey, x_curr.shape)
x_hat = x_hat * (1 - condition_mask) + condition_value * condition_mask # Apply conditioning.
# Euler step.
denoised = sde.denoise(pipe.model, x_hat, t_hat[...,None], **model_extras)

In [34]:
x_hat

Array([[[  24.986322  ],
        [  26.200941  ]],

       [[   6.626578  ],
        [ -86.615074  ]],

       [[ -61.778755  ],
        [ -50.768364  ]],

       [[  98.112816  ],
        [-118.96119   ]],

       [[ -63.429516  ],
        [  44.249477  ]],

       [[ -94.84317   ],
        [  78.15275   ]],

       [[ -35.076023  ],
        [ -26.380476  ]],

       [[  26.60377   ],
        [ -52.217564  ]],

       [[ -96.41697   ],
        [ -70.904655  ]],

       [[-168.70697   ],
        [ -12.402828  ]],

       [[ -52.63456   ],
        [ -53.060318  ]],

       [[  -2.6689637 ],
        [ -71.674324  ]],

       [[   6.1693435 ],
        [ -72.785835  ]],

       [[ 102.08415   ],
        [ -32.13413   ]],

       [[ -79.99401   ],
        [   1.3873582 ]],

       [[  32.363346  ],
        [ -85.70594   ]],

       [[  82.933     ],
        [ -53.478436  ]],

       [[  -6.2345495 ],
        [  96.64176   ]],

       [[ 160.25162   ],
        [  -5.648023  ]],

       [[  2